
# Конвертер (v5) с точечным переименованием офисов + явные города

Основано на вашем **рабочем v5** (логика парсинга не менялась).
Добавлено:
1. Пост-переименование офисов в колонке `city` после парсинга:
   - `Касса на Кропоткина-ООО` → **Кропоткина**
   - `ООО Касса Станиславского 2` → **Станиславского**
2. Явное распознавание «голых» заголовков городов в A: **Новокузнецк**, **Барнаул**.

На вход: сырой Excel (лист **Сентябрь**).  
На выход: Excel с одним листом **Debug_Parsed** (`city, program, num, fio, sum`).


In [9]:

import os

SRC_PATH = r"Сверка 3 мес.xlsx"
SRC_SHEET = "Сентябрь"

base_dir  = os.path.dirname(os.path.abspath(SRC_PATH))
base_name = os.path.splitext(os.path.basename(SRC_PATH))[0]
OUT_PATH  = os.path.join(base_dir, f"{base_name}_DEBUG_ONLY_v2.xlsx")



In [10]:

# === Utils (исходный v5 + минимальные дополнения) ===
import pandas as pd
import numpy as np
import re

def to_str(x):
    if x is None: return ""
    if isinstance(x, float) and np.isnan(x): return ""
    return str(x)

def clean_space(s: str) -> str:
    s = s.replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_index_number(val) -> bool:
    s = clean_space(to_str(val))
    return bool(re.fullmatch(r"\d+(?:\.\d+)?", s))

def parse_money(val):
    if val is None or (isinstance(val,float) and np.isnan(val)): 
        return np.nan
    s = str(val)
    s = s.replace("\xa0"," ").replace(" ","")
    s = s.replace("руб","").replace("р.","").replace("₽","")
    s = s.replace(",", ".")
    s = re.sub(r"[^0-9\.\-]", "", s)
    try:
        return float(s) if s not in ("",".") else np.nan
    except:
        try:
            return pd.to_numeric(val, errors="coerce")
        except:
            return np.nan

def looks_like_fio(b: str) -> bool:
    if not b: return False
    s = clean_space(b)
    if len(s.split()) < 2:
        return False
    return bool(re.search(r"[А-Яа-яЁё]", s))

def cap_fio(s: str) -> str:
    if not s: return s
    parts = re.split(r"\s+", s.strip())
    return " ".join(p.capitalize() for p in parts if p)

# --- header classifiers (strict) ---
CITY_MARKERS = [
    "касса", "г ", "г.", "ул", "улица", "пр.", "просп", "к.", "корп", "д.", "район",
    "пос.", "п.", "с.", "обл", "край", "город", "микрорайон", "площадь", "пер.", "переул", "ш.", "шоссе"
]
PROGRAM_MARKERS = [
    "птс", "авто", "автоптс", "залог", "ипот", "кредит", "онлайн", "налич", "карта",
    "стандарт", "exp", "express", "экспресс", "vip", "проект", "программа", "прод", "авто птс"
]

# === NEW: explicit bare city titles that should be treated as city headers ===
EXTRA_CITY_TITLES = {"новокузнецк", "барнаул"}

def is_city_header(text: str) -> bool:
    t = clean_space(text).lower()
    if t == "": return False
    if t in EXTRA_CITY_TITLES:   # <— добавлено
        return True
    if any(k in t for k in ["итог", "всего", "subtotal", "total"]): 
        return False
    if any(k in t for k in PROGRAM_MARKERS):
        return False
    return any(k in t for k in CITY_MARKERS)

def is_program_header(text: str) -> bool:
    t = clean_space(text).lower()
    if t == "": return False
    if any(k in t for k in ["итог", "всего", "subtotal", "total"]): 
        return False
    return any(k in t for k in PROGRAM_MARKERS)

# === Read Source ===
src = pd.read_excel(SRC_PATH, sheet_name=SRC_SHEET, header=None, dtype=object)

A = src.iloc[:,0] if src.shape[1] > 0 else pd.Series([], dtype=object)
B = src.iloc[:,1] if src.shape[1] > 1 else pd.Series([], dtype=object)
E = src.iloc[:,4] if src.shape[1] > 4 else pd.Series([], dtype=object)

rows = []
current_city = None
current_program = None

for i in range(src.shape[0]):
    a_raw = A.iloc[i] if i < len(A) else ""
    b_raw = B.iloc[i] if i < len(B) else ""
    e_raw = E.iloc[i] if i < len(E) else None

    a = clean_space(to_str(a_raw))
    b = clean_space(to_str(b_raw))
    e = e_raw

    a_is_index = is_index_number(a)
    b_is_fio = looks_like_fio(b)

    # --- LOAN ROW ---
    if a_is_index and b_is_fio:
        rows.append({
            "city": current_city,
            "program": current_program,
            "num": a,
            "fio": cap_fio(b),
            "sum": parse_money(e)
        })
        continue

    # --- HEADER IN A (text, not index) AND B EMPTY (no FIO) ---
    if (a != "") and (not a_is_index) and (not b_is_fio):
        if is_city_header(a) or current_city is None:
            current_city = a
            current_program = None
            continue
        if is_program_header(a) or (current_city is not None and current_program is None):
            current_program = a
            continue
        if current_city is not None:
            current_program = a
            continue
        # иначе игнорируем нерелевантное

parsed = pd.DataFrame(rows, columns=["city","program","num","fio","sum"])

# === ONLY post-rename office names in `city` ===
def rename_offices(val: str) -> str:
    if not isinstance(val, str): 
        return val
    low = val.lower()
    if ("касса на кропоткина-ооо" in low) or ("кропоткина - ооо" in low):
        return "Кропоткина"
    if ("ооо касса станиславского" in low) or ("станиславского 2" in low):
        return "Станиславского"
    return val

parsed["city"] = parsed["city"].apply(rename_offices)

# === Save ONE-SHEET file with ONLY Debug_Parsed ===
with pd.ExcelWriter(OUT_PATH, engine="xlsxwriter") as wr:
    parsed.to_excel(wr, sheet_name="Debug_Parsed", index=False)

print(f"Saved: {OUT_PATH}")


Saved: c:\MAIN\Бизнес\Аудит_оптимизация\Клиенты\Ваш капитал\API\Платежи\1С\Сверка 3 мес_DEBUG_ONLY_v2.xlsx
